In [1]:
import pandas as pd
import numpy as np

# ---------- Settings ----------
FILE = "asx200_historical.csv"
DATE_COL, CLOSE_COL = "Date", "Close"
WINDOW = 20      # moving-average length (trading sessions)
REPORT_N = 90    # latest sessions to report (selected AFTER the MA is calculated)

# Documented ASX closures for this PoC (user-supplied; confirm against the ASX calendar)
HOLIDAYS = {pd.Timestamp(k): v for k, v in {
    "2025-12-25": "Christmas Day",
    "2025-12-26": "Boxing Day",
    "2026-01-01": "New Year's Day",
    "2026-01-26": "Australia Day",
    "2026-04-03": "Good Friday",
    "2026-04-06": "Easter Monday",
    "2026-06-08": "King's Birthday",
}.items()}

# ---------- Stage 1: load and inspect ----------
raw = pd.read_csv(FILE, sep=None, engine="python")   # auto-detects comma or tab
print("Columns:", list(raw.columns))
print(raw.dtypes)
print("Rows loaded:", len(raw))



Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']
Date             object
Open            float64
High            float64
Low             float64
Close           float64
Volume            int64
Dividends         int64
Stock Splits      int64
dtype: object
Rows loaded: 255


In [2]:
# ---------- Stage 2: check required columns ----------
missing_cols = [c for c in (DATE_COL, CLOSE_COL) if c not in raw.columns]
if missing_cols:
    raise ValueError(f"Missing required column(s): {missing_cols}")



In [3]:
# ---------- Stage 3: parse dates and closes (originals kept for the log) ----------
df = pd.DataFrame({
    "csv_line": raw.index + 2,                        # line number in the file (header = line 1)
    "date_text": raw[DATE_COL].astype(str),
    "close_raw": raw[CLOSE_COL],
})
# Keep only YYYY-MM-DD so the timezone offset cannot shift the date
df["date"] = pd.to_datetime(df["date_text"].str[:10], format="%Y-%m-%d", errors="coerce")
df["close"] = pd.to_numeric(df["close_raw"], errors="coerce")



In [4]:
# ---------- Stage 4: classify each source row as accepted or rejected ----------
df["reason"] = None                                   # stays None if the row is accepted

def reject(mask, reason):
    """Give a rejection reason to rows not already rejected."""
    df.loc[mask & df["reason"].isna(), "reason"] = reason

reject(df["date"].isna(), "Invalid date")
reject(df["close"].isna(), "Missing or non-numeric close")
reject(df["close"] <= 0, "Close is zero or negative")
reject(df["date"].dt.weekday >= 5, "Source row on a weekend")
reject(df["date"].isin(list(HOLIDAYS)), "Source row on a listed ASX holiday")

# Duplicate dates (among rows still accepted)
ok = df["reason"].isna()
n_closes = df[ok].groupby("date")["close"].nunique()
conflict = n_closes[n_closes > 1].index
reject(ok & df["date"].isin(conflict), "Duplicate date with conflicting closes")
ok = df["reason"].isna()
dup_rows = df[ok].duplicated("date", keep="first")
df.loc[dup_rows[dup_rows].index, "reason"] = "Duplicate date, identical close (first row kept)"

accepted = df[df["reason"].isna()]
rejected = df[df["reason"].notna()]
closes = accepted.set_index("date")["close"].sort_index()   # one close per date, chronological



In [5]:
# ---------- Stage 5: classify every calendar date in the source date range ----------
all_days = pd.date_range(df["date"].min(), df["date"].max())
non_trading = {}                                      # date -> reason (not sessions, not errors)
for d in all_days:
    if d.weekday() >= 5:
        non_trading[d] = "Weekend"
    elif d in HOLIDAYS:
        non_trading[d] = HOLIDAYS[d]
expected_sessions = all_days.difference(pd.DatetimeIndex(list(non_trading)))

# Unresolved missing sessions: expected session with no accepted observation
rejected_by_date = rejected.dropna(subset=["date"]).groupby("date")["reason"].first()
unresolved = {}
for d in expected_sessions.difference(closes.index):
    unresolved[d] = ("Source row rejected: " + rejected_by_date[d]) if d in rejected_by_date.index \
                    else "No source row - investigate"


In [6]:
# ---------- Stage 6: 20-session moving average on ALL available data ----------
# Place closes on the expected-session calendar; unresolved sessions become NaN,
# so no window can bridge them. Holidays/weekends are not in the calendar, so
# windows spanning them remain valid.
series = closes.reindex(expected_sessions)
series.index.name = "date"
result = pd.DataFrame({"close": series})
result["ma20"] = series.rolling(WINDOW, min_periods=WINDOW).mean()   # NaN = unavailable

# ---------- Stage 7: display latest results (latest 90 chosen AFTER calculation) ----------
report = result.tail(REPORT_N)
print(f"\nLatest 10 trading days (of the latest {REPORT_N} reported):")
print(report.tail(10).round(2).to_string())
# print(report.round(2).to_string())                 # uncomment to display all 90 rows




Latest 10 trading days (of the latest 90 reported):
             close     ma20
date                       
2026-10-09  8716.6  8710.33
2026-10-12  8783.2  8712.00
2026-10-13  8783.9  8717.57
2026-10-14  8727.9  8719.14
2026-10-15  8719.7  8718.50
2026-10-16  8806.7  8722.28
2026-10-19  8845.7  8727.97
2026-10-20  8816.9  8730.92
2026-10-21  8831.3  8734.22
2026-10-22  8793.2  8738.78


In [9]:
# ---------- Quality log ----------
valid_ma = result["ma20"].notna()
window_has_gap = series.isna().rolling(WINDOW, min_periods=1).sum() > 0
ma_blocked = int((result["ma20"].isna() & window_has_gap).sum())     # blocked by unresolved session
ma_start = int(result["ma20"].isna().sum()) - ma_blocked             # fewer than 20 sessions so far
n_weekend = sum(v == "Weekend" for v in non_trading.values())

print("\n=== QUALITY LOG ===")
print("Rows loaded:", len(raw))
print("Date range (source dates):", df["date"].min().date(), "to", df["date"].max().date())

print("\nAccepted records:", len(accepted))

print("\nRejected source records:", len(rejected))
for r in rejected.itertuples():
    print(f"  - line {r.csv_line}: date='{r.date_text}', close='{r.close_raw}' -> {r.reason}")

print(f"\nConfirmed non-trading dates: {len(non_trading)} "
      f"({n_weekend} weekend, {len(non_trading) - n_weekend} holiday)")
for d, why in non_trading.items():
    if why != "Weekend":                              # weekends counted, not listed, to keep output short
        print(f"  - {d.date()}: {why}")

print("\nUnresolved missing sessions:", len(unresolved))
for d, why in unresolved.items():
    print(f"  - {d.date()}: {why}")

print("\nValid MA results:", int(valid_ma.sum()))
print("MA unavailable - window includes an unresolved session:", ma_blocked)
print("MA unavailable - fewer than 20 sessions at start of series:", ma_start)

if valid_ma.sum() == 0:
    status = "ISSUE: no valid moving-average results"
elif len(rejected) or len(unresolved):
    status = "COMPLETED WITH ISSUES - review rejected records / unresolved sessions"
else:
    status = "SUCCESS"
print("Status:", status)


=== QUALITY LOG ===
Rows loaded: 255
Date range (source dates): 2025-10-22 to 2026-10-22

Accepted records: 255

Rejected source records: 0

Confirmed non-trading dates: 111 (104 weekend, 7 holiday)
  - 2025-12-25: Christmas Day
  - 2025-12-26: Boxing Day
  - 2026-01-01: New Year's Day
  - 2026-01-26: Australia Day
  - 2026-04-03: Good Friday
  - 2026-04-06: Easter Monday
  - 2026-06-08: King's Birthday

Unresolved missing sessions: 0

Valid MA results: 236
MA unavailable - window includes an unresolved session: 0
MA unavailable - fewer than 20 sessions at start of series: 19
Status: SUCCESS
